# Aggregate Classification Sweep Results

Load classification sweep results for multiple campaigns, select one top percentage, and average the results across campaigns.

In [7]:
from pathlib import Path

import pandas as pd

## Parameters

In [8]:
Campaigns = ["Iran_1", "UAE", "Venezuela"] # "China_1", "Cuba", "Russia_1"
top_percentage = 100

In [9]:
def find_project_root(start_path):
    """Find the project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the project root containing AGENTS.md.")


def get_results_path(project_root, campaign):
    """Build the classification sweep results path for a campaign.

    Args:
        project_root: Root directory of the project.
        campaign: Campaign directory name.

    Returns:
        Path to the campaign's classification sweep CSV file.
    """
    return (
        project_root
        / "results"
        / "Labeled_Datasets"
        / campaign
        / f"{campaign}_part_all"
        / "step_3_key_authors_graph"
        / "classification_sweep"
        / "classification_sweep_results.csv"
    )

In [10]:
if not Campaigns:
    raise ValueError("Campaigns must contain at least one campaign name.")

project_root = find_project_root(Path.cwd())
campaign_results = []

for campaign in Campaigns:
    results_path = get_results_path(project_root, campaign)
    campaign_df = pd.read_csv(results_path)
    campaign_df = campaign_df.loc[
        campaign_df["top_percentage"].eq(top_percentage)
    ]
    campaign_results.append(campaign_df)

all_results_df = pd.concat(campaign_results, ignore_index=True)
all_results_df["indicators_configuration"] = all_results_df[
    "indicators_configuration"
].str.replace(
    " (Unweighted Clustering, No Square Clustering)",
    "",
    regex=False,
)
group_columns = all_results_df.select_dtypes(exclude="number").columns.tolist()
aggregated_df = (
    all_results_df.groupby(group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(group_columns)
    .reset_index(drop=True)
)

display(aggregated_df)

classification_group_columns = [
    "topic_col",
    "account_relations_configuration",
    "indicators_configuration",
]
classification_metric_columns = [
    column
    for column in all_results_df.columns
    if column.startswith(("graph_authors_", "all_authors_"))
]
classification_metrics_df = (
    all_results_df[
        classification_group_columns + classification_metric_columns
    ]
    .groupby(classification_group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(classification_group_columns)
    .reset_index(drop=True)
)

display(classification_metrics_df)

summary_metric_columns = {
    "all_authors_auc": "ROC_AUC",
    "all_authors_auc_pr": "PR_AUC",
}

topic_col_aggregation_df = (
    all_results_df.groupby("topic_col", as_index=False, dropna=False)[
        list(summary_metric_columns)
    ]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("topic_col")
    .reset_index(drop=True)
)
display(topic_col_aggregation_df)

account_relations_aggregation_df = (
    all_results_df.groupby(
        "account_relations_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns)]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("account_relations_configuration")
    .reset_index(drop=True)
)
display(account_relations_aggregation_df)

indicators_aggregation_df = (
    all_results_df.groupby(
        "indicators_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns)]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("indicators_configuration")
    .reset_index(drop=True)
)
display(indicators_aggregation_df)

,topic_col,account_relations_configuration,indicators_configuration,model,top_percentage,indicators_runtime_seconds,n_graph_authors,n_all_authors,graph_authors_auc,graph_authors_f1,graph_authors_auc_pr,graph_authors_recall,graph_authors_precision,all_authors_auc,all_authors_f1,all_authors_auc_pr,all_authors_recall,all_authors_precision
0,BERTopic_topic_10,interactions,All,logistic_regression,100.0,50.207526,10077.666667,10077.666667,0.898117,0.712021,0.781755,0.591521,0.903547,0.898117,0.712021,0.781755,0.591521,0.903547
1,BERTopic_topic_10,interactions,All,random_forest,100.0,50.207526,10077.666667,10077.666667,0.952737,0.872080,0.910051,0.794897,0.970860,0.952737,0.872080,0.910051,0.794897,0.970860
2,BERTopic_topic_10,interactions,All,xgboost,100.0,50.207526,10077.666667,10077.666667,0.957441,0.888158,0.914599,0.826851,0.962329,0.957441,0.888158,0.914599,0.826851,0.962329
3,BERTopic_topic_10,interactions,All but Betweenness and Harmonic,logistic_regression,100.0,50.207526,10077.666667,10077.666667,0.897433,0.716294,0.779653,0.595505,0.903654,0.897433,0.716294,0.779653,0.595505,0.903654
4,BERTopic_topic_10,interactions,All but Betweenness and Harmonic,random_forest,100.0,50.207526,10077.666667,10077.666667,0.951026,0.875151,0.908795,0.800168,0.970246,0.951026,0.875151,0.908795,0.800168,0.970246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,K_Means_topic_60,text_similarity+interactions,All but Betweenness and Harmonic,random_forest,100.0,1842.058740,10356.000000,10356.000000,0.988239,0.954817,0.971305,0.916530,0.996441,0.988239,0.954817,0.971305,0.916530,0.996441
437,K_Means_topic_60,text_similarity+interactions,All but Betweenness and Harmonic,xgboost,100.0,1842.058740,10356.000000,10356.000000,0.988773,0.956081,0.970965,0.926350,0.987784,0.988773,0.956081,0.970965,0.926350,0.987784
438,K_Means_topic_60,text_similarity+interactions,Core Number Only,logistic_regression,100.0,1842.058740,10356.000000,10356.000000,0.727644,0.000000,0.131395,0.000000,0.000000,0.727644,0.000000,0.131395,0.000000,0.000000
439,K_Means_topic_60,text_similarity+interactions,Core Number Only,random_forest,100.0,1842.058740,10356.000000,10356.000000,0.903335,0.427632,0.449099,0.319149,0.647841,0.903335,0.427632,0.449099,0.319149,0.647841


,topic_col,account_relations_configuration,indicators_configuration,graph_authors_auc,graph_authors_f1,graph_authors_auc_pr,graph_authors_recall,graph_authors_precision,all_authors_auc,all_authors_f1,all_authors_auc_pr,all_authors_recall,all_authors_precision
0,BERTopic_topic_10,interactions,All,0.936098,0.824086,0.868802,0.737756,0.945579,0.936098,0.824086,0.868802,0.737756,0.945579
1,BERTopic_topic_10,interactions,All but Betweenness and Harmonic,0.934495,0.826170,0.866893,0.740419,0.944994,0.934495,0.826170,0.866893,0.740419,0.944994
2,BERTopic_topic_10,interactions,Core Number Only,0.773825,0.360028,0.448062,0.276603,0.534875,0.773825,0.360028,0.448062,0.276603,0.534875
3,BERTopic_topic_10,similarity,All,0.979419,0.912130,0.953277,0.864945,0.971856,0.979419,0.912130,0.953277,0.864945,0.971856
4,BERTopic_topic_10,similarity,All but Betweenness and Harmonic,0.978537,0.907629,0.947953,0.859013,0.970477,0.978537,0.907629,0.947953,0.859013,0.970477
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,K_Means_topic_60,text_similarity,All but Betweenness and Harmonic,0.955031,0.843580,0.876459,0.771959,0.965742,0.955031,0.843580,0.876459,0.771959,0.965742
143,K_Means_topic_60,text_similarity,Core Number Only,0.841412,0.260367,0.332203,0.187125,0.434084,0.841412,0.260367,0.332203,0.187125,0.434084
144,K_Means_topic_60,text_similarity+interactions,All,0.962468,0.894886,0.912134,0.829242,0.986106,0.962468,0.894886,0.912134,0.829242,0.986106
145,K_Means_topic_60,text_similarity+interactions,All but Betweenness and Harmonic,0.952982,0.840343,0.872472,0.765412,0.972295,0.952982,0.840343,0.872472,0.765412,0.972295


,topic_col,ROC_AUC,PR_AUC
0,BERTopic_topic_10,0.945314,0.859693
1,BERTopic_topic_128,0.945265,0.852223
2,BERTopic_topic_256,0.943722,0.839869
3,BERTopic_topic_512,0.947240,0.848214
4,K_Means_topic_128,0.958964,0.913980
5,K_Means_topic_16,0.959994,0.922793
6,K_Means_topic_60,0.915387,0.704337


,account_relations_configuration,ROC_AUC,PR_AUC
0,interactions,0.881485,0.722514
1,similarity,0.952337,0.875834
2,similarity+interactions,0.951888,0.875081
3,similarity+text_similarity,0.958277,0.867989
4,similarity+text_similarity+interactions,0.957553,0.868031
5,text_similarity,0.957471,0.867816
6,text_similarity+interactions,0.957841,0.868586


,indicators_configuration,ROC_AUC,PR_AUC
0,All,0.969105,0.930101
1,All but Betweenness and Harmonic,0.971676,0.929772
2,Core Number Only,0.895012,0.688348


In [11]:
output_dir = project_root / "results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated"
output_dir.mkdir(parents=True, exist_ok=True)

campaigns_name = "_".join(Campaigns).lower().replace(" ", "_")
percentage_name = f"{top_percentage:g}"
output_stem = (
    f"{campaigns_name}_top_percentage_{percentage_name}"
    "_aggregated_classification_sweep_results"
)
csv_path = output_dir / f"{output_stem}.csv"
latex_path = output_dir / f"{output_stem}.tex"
metrics_output_stem = (
    f"{campaigns_name}_top_percentage_{percentage_name}"
    "_aggregated_classification_metrics"
)
metrics_csv_path = output_dir / f"{metrics_output_stem}.csv"
metrics_latex_path = output_dir / f"{metrics_output_stem}.tex"

aggregated_df.to_csv(csv_path, index=False)
aggregated_df.to_latex(latex_path, index=False, float_format="%.4f")
classification_metrics_df.to_csv(metrics_csv_path, index=False)
classification_metrics_df.to_latex(
    metrics_latex_path, index=False, float_format="%.4f"
)

summary_aggregations = {
    "topic_col": topic_col_aggregation_df,
    "account_relations_configuration": account_relations_aggregation_df,
    "indicators_configuration": indicators_aggregation_df,
}
for aggregation_name, summary_df in summary_aggregations.items():
    summary_output_stem = f"{output_stem}_by_{aggregation_name}"
    summary_csv_path = output_dir / f"{summary_output_stem}.csv"
    summary_latex_path = output_dir / f"{summary_output_stem}.tex"
    summary_df.to_csv(summary_csv_path, index=False)
    summary_df.to_latex(
        summary_latex_path, index=False, float_format="%.4f"
    )
    print(f"Saved summary CSV: {summary_csv_path}")
    print(f"Saved summary LaTeX: {summary_latex_path}")

print(f"Saved CSV: {csv_path}")
print(f"Saved LaTeX: {latex_path}")
print(f"Saved classification metrics CSV: {metrics_csv_path}")
print(f"Saved classification metrics LaTeX: {metrics_latex_path}")

Saved summary CSV: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/iran_1_uae_venezuela_top_percentage_100_aggregated_classification_sweep_results_by_topic_col.csv
Saved summary LaTeX: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/iran_1_uae_venezuela_top_percentage_100_aggregated_classification_sweep_results_by_topic_col.tex
Saved summary CSV: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/iran_1_uae_venezuela_top_percentage_100_aggregated_classification_sweep_results_by_account_relations_configuration.csv
Saved summary LaTeX: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_intra_classification/ aggregated/iran_1_uae_venezuela_top_percentage_100_aggregated_classification_sweep_results_by_account_relations_configuration.tex
Saved summary CSV: /home/amitner/Backbone-Graph/results/Labeled_Datasets/step_4_inter_in